# Audit shared-drive disk usage

The microscope PC's data drives are shared, laid out as
`{root}/{lab_member}/{sample_dir}`. Measures the size and creation date of
every `sample_dir` under the listed roots and sorts them oldest-first and
largest-first, to see who to ask to free space. Logic in `MERci.disk_audit`.

In [ ]:
import os
import sys
from pathlib import Path

MERCI_DIR = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/misc/)
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.disk_audit import audit_disk_usage, group_by_lab_member

print(f"MERCI_DIR : {MERCI_DIR}")

## Roots to scan

Each root holds one folder per lab member, each with one folder per sample.
A root can be a drive (`"D:/"`) or a folder (`"E:/archive/"`). Missing roots
are skipped with a warning.

In [ ]:
ROOTS = [
    "D:/",
    "E:/",
    "F:/",
]

## Run the scan

Walks every file under every root, so it can take minutes; prints progress per
sample folder.

In [ ]:
df = audit_disk_usage(ROOTS)
print(f"\nFound {len(df)} sample folder(s).")
df

## Size threshold

The sorted views show only folders of at least `MIN_SIZE_GB` (`0` = all).

In [ ]:
MIN_SIZE_GB = 0   # e.g. 50 to only show sample folders of 50 GB or more

filtered_df = df[df["size_gb"] >= MIN_SIZE_GB] if not df.empty else df
print(f"{len(filtered_df)} of {len(df)} sample folder(s) are >= {MIN_SIZE_GB} GB.")

## Sorted by age -- oldest first

`created` is the folder's creation time on this disk (`st_ctime`).
`earliest_file_modified`/`latest_file_modified` are its oldest/newest file
`mtime`, a cross-check if the folder was renamed later.

In [ ]:
cols = ["root", "lab_member", "sample_dir", "size_gb", "created",
        "days_since_created", "n_files"]

if filtered_df.empty:
    print("No sample folders found -- check ROOTS and MIN_SIZE_GB above.")
else:
    display(filtered_df.sort_values("created")[cols].reset_index(drop=True))

## Sorted by size -- largest first

In [ ]:
if filtered_df.empty:
    print("No sample folders found -- check ROOTS and MIN_SIZE_GB above.")
else:
    display(filtered_df.sort_values("size_gb", ascending=False)[cols].reset_index(drop=True))

## Per lab member -- one table each

One table per lab member (largest first), with folder count and total size, to
paste into an email. Names are grouped ignoring case; the header lists every
spelling found.

In [ ]:
if filtered_df.empty:
    print("No sample folders found -- check ROOTS and MIN_SIZE_GB above.")
else:
    for display_name, group in group_by_lab_member(filtered_df):
        print(f"\n{display_name}  --  {len(group)} folder(s), {group['size_gb'].sum():.1f} GB total")
        display(group.sort_values("size_gb", ascending=False)[cols].reset_index(drop=True))

## Optional: save the full table to CSV

Uncomment to save a timestamped CSV.

In [ ]:
# import datetime
# out_path = MERCI_DIR / f"disk_usage_audit_{datetime.datetime.now():%Y%m%d_%H%M}.csv"
# df.sort_values("created").to_csv(out_path, index=False)          # full scan
# # filtered_df.sort_values("created").to_csv(out_path, index=False)  # only >= MIN_SIZE_GB
# print(f"Saved: {out_path}")